# 第一部分：分词
## 一、 加载分词器

AutoTokenizer

在 `transformers` 库中，`AutoTokenizer` 是一个非常智能的类。它会根据你提供的模型路径，自动识别并加载对应的分词配置文件（如 `tokenizer_config.json` 和 `vocab.json`）。

#### 1. 关键参数解析

- **`pretrained_model_name_or_path`**:
    - **含义**：本地存放模型的文件夹路径。        
    - **工程意义**：告诉程序去哪里读取那本“词表密码本”。
- **`trust_remote_code=True`**:
    - **含义**：允许执行模型仓库中的自定义分词代码。
    - **必要性**：很多国产模型（如 Qwen）使用了特殊的编码逻辑，必须开启此项才能正确加载。


In [6]:
from transformers import AutoTokenizer

# 1. 设置模型路径（请确保路径下有 tokenizer.json 等文件）
model_dir = "C:\\0store\\LLM\\Qwen2.5-1.5B-Instruct" # 替换为你实际的下载路径

# 2. 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    model_dir, 
    trust_remote_code=True
)


### 查看特殊 Token (Special Tokens)

在模型文件夹的 `tokenizer_config.json` 中定义了该模型专属的信号灯。我们可以通过以下属性直接调用查看。

- **常用属性**：
    
    - `tokenizer.bos_token` / `bos_token_id`: 句子开始标记。
        
    - `tokenizer.eos_token` / `eos_token_id`: 句子结束标记。
        
    - `tokenizer.pad_token` / `pad_token_id`: 填充标记

In [14]:
print(f"--- 3. 特殊 Token 一览 ---")
print(f"结束符 (EOS): {tokenizer.eos_token} -> ID: {tokenizer.eos_token_id}")
print(f"填充符 (PAD): {tokenizer.pad_token} -> ID: {tokenizer.pad_token_id}")
# 注意：有些模型可能没有定义 BOS，会返回 None

--- 3. 特殊 Token 一览 ---
结束符 (EOS): <|im_end|> -> ID: 151645
填充符 (PAD): <|endoftext|> -> ID: 151643


## 实战：Tokenizer 的四步进阶用法

### 1. 第一步：分词 (Tokenize)
使用 tokenize 函数将字符串切分为最小语义单元（Tokens）。

- 函数：tokenizer.tokenize(text)

- 输入：text (String) —— 原始人类语言文本。

- 输出：List[str] —— 切分后的子词列表。

In [12]:
# 步骤 1: 仅分词，不转 ID
text = "I'm fine-tuning a 1.5B LLM! It's 'awesome', isn't it?"

# 步骤 1: 观察子词切分
tokens = tokenizer.tokenize(text)
print(f"类型: {type(tokens)}")
print(f"分词结果:\n{tokens}")
# 你会发现 'fine-tuning' 可能会被切分为 ['fine', '-', 't', 'uning']
# 'isn't' 可能会被切分为 ['isn', "'t"]

类型: <class 'list'>
分词结果:
['I', "'m", 'Ġfine', '-t', 'uning', 'Ġa', 'Ġ', '1', '.', '5', 'B', 'ĠL', 'LM', '!', 'ĠIt', "'s", "Ġ'", 'awesome', "',", 'Ġisn', "'t", 'Ġit', '?']


### 2. 第二步：编号 (Convert Tokens to IDs)

将上一步得到的子词列表映射为词表中的整数索引。

- **函数**：`tokenizer.convert_tokens_to_ids(tokens)`
    
- **输入**：`List[str]` —— 上一步产生的 Token 列表。
    
- **输出**：`List[int]` —— 对应的整数 ID 列表。

In [13]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"--- 2. 编号结果 ---")
print(f"类型: {type(ids)}")
print(f"内容: {ids}")
# 每一个数字都对应词表中唯一的一个高维向量索引

--- 2. 编号结果 ---
类型: <class 'list'>
内容: [40, 2776, 6915, 2385, 37202, 264, 220, 16, 13, 20, 33, 444, 10994, 0, 1084, 594, 364, 16875, 516, 4436, 944, 432, 30]


In [5]:
# 一个batch输入多个文本
texts = [
    "你好，大模型微调！",
    "下面是第二个文本"
]

inputs = tokenizer(
    texts, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=512
)

print("--- 编码结果 ---")
print(f"Input IDs 内容:\n{inputs['input_ids']}")
print(f"\nAttention Mask 内容:\n{inputs['attention_mask']}")
print(f"\n张量形状 (Shape): {inputs['input_ids'].shape}") # 应为 [1, 序列长度]

--- 编码结果 ---
Input IDs 内容:
tensor([[108386,   3837,  26288, 104949,  48934,  47872,   6313],
        [112918, 106657, 108704, 151643, 151643, 151643, 151643]])

Attention Mask 内容:
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0, 0]])

张量形状 (Shape): torch.Size([2, 7])


### 第三步：一键集成流程（Batch Processing）
在实际工程中，我们通常直接调用 `tokenizer` 对象本身（即调用其 `__call__` 方法），它可以一次性处理**列表形式**的多条文本，并自动完成 Padding 和 Masking。

- **输入参数**：
    
    - `text_list`: `List[str]` —— 包含多条文本的列表。
        
    - `padding=True`: 自动将短句补齐到当前 Batch 中最长句子的长度。
        
    - `return_tensors="pt"`: 返回 PyTorch 张量格式。


它的返回值不是一个简单的列表，而是一个 **Python 字典 (dict)**。这个字典通常包含两个核心张量：

- **`input_ids`**：文本被数字化后的整数序列（模型的“原材料”）。
    
- **`attention_mask`**：指示哪些位置是真实 Token (1)，哪些是填充的 Padding (0)（模型的“滤镜”）。

In [ ]:
# 准备一个长短不一的列表
batch_text = ["你好", "大模型微调是非常有趣的实战"]

# 确保已经设置了 pad_token，否则填充会报错
# 大多数情况下，如果没有pad_token，我们会将eos_token作为pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(batch_text, 
                   padding=True, 
                   return_tensors="pt")

print(f"--- 4. 集成处理结果 ---")
print(f"Input IDs 矩阵:\n{inputs['input_ids']}")
print(f"\nAttention Mask 矩阵:\n{inputs['attention_mask']}")

--- 4. 集成处理结果 ---
Input IDs 矩阵:
tensor([[108386, 151643, 151643, 151643, 151643, 151643, 151643],
        [ 26288, 104949,  48934,  47872, 104771, 107935, 108433]])

Attention Mask 矩阵:
tensor([[1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]])
